In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.shape

(50000, 2)

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.drop_duplicates(inplace=True) 

In [6]:
df.shape

(49582, 2)

### preprocessing

In [7]:
## lowercase

df["review"]=df["review"].str.lower()

In [8]:
## remove url

import re

def remove_url(text):
    text=re.sub(r"http\S+","",text)
    return text
df["review"]=df["review"].apply(remove_url)

In [9]:
## remove punctuation

def remove_punctuation(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
df["review"]=df["review"].apply(remove_punctuation)

In [10]:
# remove html

def remove_html(text):
    text=re.sub(r"<.*?>","",text)
    return text
df["review"]=df["review"].apply(remove_html)

In [11]:
!pip install nltk


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
## removing stopwards

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords") 

[nltk_data] Downloading package punkt to C:\Users\Ritvika
[nltk_data]     Jain\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Ritvika
[nltk_data]     Jain\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Ritvika
[nltk_data]     Jain\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopword(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")
    return text

df["review"]=df["review"].apply(remove_stopword)

In [15]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [16]:
# stemming
from nltk.stem import PorterStemmer

def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]
    
    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)

df["review"]=df["review"].apply(stemming)

In [17]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [18]:
# encoding

from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

In [19]:
y=df["sentiment"]

In [20]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


In [21]:
# vectorization

from sklearn.feature_extraction.text import TfidfVectorizer 

tf=TfidfVectorizer(max_features=5000)

x=tf.fit_transform(df["review"])

In [32]:
## dataset and dataloader

from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42)

In [33]:
x_train.shape

(39665, 5000)

In [34]:
y_train.shape 

(39665,)

In [35]:
import torch
from torch.utils.data import TensorDataset ,DataLoader

In [36]:
x_train=x_train.toarray()
x_test=x_test.toarray()

In [37]:
train_set=TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set=TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [39]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

In [41]:
# build RNN

import torch.nn as nn
import torch.optim as optimizer

In [43]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size=hidden_size
        self.num_layers=num_layers
        
        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
        self.fc=nn.Linear(hidden_size,1)

    def forward(self,x):
        h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)
        out,_=self.rnn(x,h0)

        out=self.fc(out[:,-1,:])
        return out

In [46]:
input_size=x_train.shape[1]
model=RNN(input_size)

criterion=nn.BCELoss()
optimizer=optimizer.Adam(model.parameters())

In [47]:
## training

epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:

        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)   # Add singleton dimension

        outputs = model(Xb)    # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze())   # (batch_size,) => probability

        loss = criterion(outputs, yb)

        loss.backward()

        optimizer.step()

    print(f"Epoch = {epoch+1}/{epochs} and Loss = {loss.item()}")

Epoch = 1/10 and Loss = 0.2502354085445404
Epoch = 2/10 and Loss = 0.23398979008197784
Epoch = 3/10 and Loss = 0.1755436807870865
Epoch = 4/10 and Loss = 0.2701456546783447
Epoch = 5/10 and Loss = 0.27019086480140686
Epoch = 6/10 and Loss = 0.23611041903495789
Epoch = 7/10 and Loss = 0.16801880300045013
Epoch = 8/10 and Loss = 0.17233632504940033
Epoch = 9/10 and Loss = 0.2678595185279846
Epoch = 10/10 and Loss = 0.21354274451732635


In [48]:
# eval

model.eval()

with torch.no_grad():

    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:

        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)

        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)

        correct_vals += (predicted == yb).sum().item()

print(f"Accuracy = {correct_vals / tot_vals * 100:.2f}%")
        

Accuracy = 85.66%
